# datalake parquet 내용 확인

셀 왼쪽의 ▷ 버튼을 위에서부터 순서대로 누르면 됩니다.
마지막 셀 실행 후 오른쪽 위 **Jupyter: Variables** 패널을 열면 `df`가 보이고,
그 옆 돋보기 아이콘을 누르면 엑셀처럼 정렬/필터 가능한 표(Data Viewer)로 전체 내용을 볼 수 있습니다.

In [ ]:
from pathlib import Path
import pandas as pd

ENV_PATH = Path.cwd().parent / ".env"

def load_env(path: Path) -> dict[str, str]:
    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
    return env

env = load_env(ENV_PATH)
storage_options = {
    "key": env["MINIO_ROOT_USER"],
    "secret": env["MINIO_ROOT_PASSWORD"],
    "client_kwargs": {"endpoint_url": "http://localhost:9000"},
}

In [ ]:
# path만 바꿔서 다른 레이어/테이블도 확인 가능
# 예: "bronze/steamspy_all", "raw/steam/steamspy_all"
path = "silver/steamspy_all"

df = pd.read_parquet(f"s3://datalake/{path}/", storage_options=storage_options)
df.shape

In [ ]:
# 이 셀 실행하면 아래에 표로 렌더링됨 (Variables 패널의 돋보기 아이콘으로 전체 보기도 가능)
df

## bronze(필터링 전 원본)와 비교

silver에서 몇 건이 걸러졌는지 확인용.

In [ ]:
df_bronze = pd.read_parquet(
    "s3://datalake/bronze/steamspy_all/", storage_options=storage_options
)
print(f"bronze : {len(df_bronze)}")
print(f"silver : {len(df)}")
print(f"걸러진 개수 : {len(df_bronze) - len(df)}")

In [ ]:
# bronze 표로 확인 (Variables 패널의 df_bronze 돋보기 아이콘으로 전체 보기도 가능)
df_bronze